# Planting-window pipeline — run in Google Colab

Crop-specific dekadal planting-window estimation (GHA / ICPAC) on Google Earth Engine.

**Before you start:** your Google account must be registered for Earth Engine and you need a
GEE **cloud project** (here `ee-manzikye`). Outputs export to **Google Drive → `planting_outputs/`**.

> ⚠️ Colab does **not** raise your compute quota — that is tied to the project, not the runtime.
> If the project is in noncommercial *restricted mode*, heavy jobs (WRSI) will still be throttled here.

## 1. Install dependencies
(Colab already has `earthengine-api`, but this pins current versions.)

In [1]:
!pip -q install "earthengine-api>=1.4.0" pyyaml pandas

## 2. Authenticate + initialise Earth Engine
Run the cell, click the link, paste the token. Set `PROJECT` to your GEE cloud project.

In [3]:
import ee

PROJECT = "ee-manzikye"   # <-- your GEE cloud project id

ee.Authenticate()          # opens an auth link the first time
ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

EE ready: ok


## 3. Get the pipeline code into Colab
Upload `planting_pipeline.zip` (the file I built for you) using the cell below, then it unzips
and `cd`s into it.

In [4]:
import zipfile, os
from google.colab import files

up = files.upload()                 # choose planting_pipeline.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall(".")
os.chdir("planting_pipeline")
print("cwd:", os.getcwd())
print("files:", sorted(os.listdir()))

ModuleNotFoundError: No module named 'google.colab'

## 4. Run one product
`run.py` starts Earth Engine batch exports (they run on Google's servers, land in Drive).
Change `--country` / `--crop` / `--year`, or drop `--country`/`--crop` to run everything viable for the year.
Add `--mask-asset users/you/your_mask` for a crop-specific mask (otherwise WorldCereal is used).

In [ ]:
os.environ["EE_PROJECT"] = PROJECT
!EE_PROJECT=$PROJECT python run.py --year 2024 --country Kenya --crop maize

## 5. Monitor the export tasks
Re-run this cell to refresh. States: PENDING → RUNNING → SUCCEEDED / FAILED / CANCELLED.

In [ ]:
seen = {}
for op in ee.data.listOperations():
    md = op.get("metadata", {})
    d = md.get("description", "")
    if "Kenya_maize" in d:                    # adjust filter to your run
        ct = md.get("createTime", "")
        if d not in seen or ct > seen[d][0]:   # keep the latest task per name
            seen[d] = (ct, md.get("state", "?"), op.get("error", {}).get("message", ""))
for d in sorted(seen):
    ct, st, err = seen[d]
    print(f"{st:10} {d}" + (f"  -> {err}" if err else ""))

## 6. Where the outputs are
When tasks show **SUCCEEDED**, files are in your **Google Drive → `planting_outputs/`**:
- `planting_<country>_<crop>_<season>_<year>` — GeoTIFF, per-pixel planting dekad
- `wrsi_<...>` — GeoTIFF (WRSI + deficit mm + crop-performance class)
- `<...>_zonal` — CSV, admin-1 modal / P10 / P50 / P90 planting dekad

You can also watch tasks at https://code.earthengine.google.com/tasks